In [2]:
import numpy as np 
import pandas as pd  

In [3]:
df = pd.read_csv(r"D:\MASTERS\FOML\assignment\Project\Project@\ML-Health-Insurance\Dataset of health insurance portfolio.xlsx - Sheet1 (1).csv")

In [4]:
df.head(10)

,ID,ID_policy,ID_insured,period,date_effect_insured,date_lapse_insured,date_effect_policy,date_lapse_policy,year_effect_insured,year_lapse_insured,...,C_H,C_GI,C_II,C_IE_P,C_IE_S,C_IE_T,C_GE_P,C_GE_S,C_GE_T,C_C
0,1_1,1,1,2017,6/15/1993,12/20/2017,6/15/1993,NaN,1993,2017.0,...,H1,99.0,85.0,18.0,52.0,59.0,4.0,14.0,96.0,C6
1,1_2,1,2,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,H1,99.0,85.0,18.0,52.0,59.0,4.0,14.0,96.0,C6
2,2_1,2,1,2017,6/16/1993,NaN,6/16/1993,NaN,1993,NaN,...,H6,100.0,94.0,26.0,11.0,85.0,7.0,1.0,100.0,C6
3,3_3,3,3,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3_5,3,5,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,H6,99.0,81.0,32.0,34.0,69.0,9.0,4.0,98.0,C6
5,3_6,3,6,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,H1,32.0,9.0,100.0,80.0,2.0,83.0,51.0,19.0,C4
6,3_7,3,7,2017,4/19/1999,NaN,6/15/1993,NaN,1999,NaN,...,H6,99.0,82.0,44.0,21.0,74.0,12.0,2.0,98.0,C6
7,4_3,4,3,2017,8/8/1995,NaN,7/1/1993,NaN,1995,NaN,...,H6,100.0,94.0,26.0,11.0,85.0,7.0,1.0,100.0,C6
8,5_1,5,1,2017,7/1/1993,NaN,7/1/1993,NaN,1993,NaN,...,H6,99.0,82.0,44.0,21.0,74.0,12.0,2.0,98.0,C6
9,5_2,5,2,2017,7/1/1993,NaN,7/1/1993,NaN,1993,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.shape

(228711, 42)

## Checking whether ID is consistant for different policies and insured id's

In [6]:
df.groupby('ID')['ID_policy'].nunique().value_counts()

ID_policy
1    100453
Name: count, dtype: int64

In [7]:
df.groupby('ID')['ID_insured'].nunique().value_counts()

ID_insured
1    100453
Name: count, dtype: int64

Each ID is associated with exactly one ID policy and ID insured , which concludes ID is suitable for renewal premium prediction.

## Checking the chronological pattern among the dataset through ID and Period 

In [8]:
df.groupby('ID')['period'].agg(lambda x: sorted(x.unique()))

ID
10000_1    [2017, 2018, 2019]
10001_1    [2017, 2018, 2019]
10002_1    [2017, 2018, 2019]
10002_3    [2017, 2018, 2019]
10002_4    [2017, 2018, 2019]
                  ...        
9999_3     [2017, 2018, 2019]
999_2      [2017, 2018, 2019]
999_3      [2017, 2018, 2019]
99_1       [2017, 2018, 2019]
9_1        [2017, 2018, 2019]
Name: period, Length: 100453, dtype: object

In [9]:
year_patterns = df.groupby('ID')['period'].agg(
    lambda x: tuple(sorted(x.unique()))
)

year_patterns.value_counts()

period
(2017, 2018, 2019)    55635
(2019,)               13842
(2017,)               12640
(2017, 2018)          10183
(2018, 2019)           6804
(2018,)                1348
(2017, 2019)              1
Name: count, dtype: int64

## Findings  
1. Some IDs are registered in between the observed period(2017,2018,2019),so they must new customers this means shorter history record is still usefull 
2. Some ID lacks the consecutive record over the given years , which wont be beneficial for preDicting the renewal premium prediction
3. Some IDs have records in particular year only , so they wont be usefull for further prediction  



## Identifying the possible IDs which are consecutive over the period (2017-2019 , 2017-2018 and 2018-2019)

In [10]:
# Sort observations by ID and year
df_sorted = df.sort_values(['ID', 'period']).copy()

# Get the following year's premium and period for each ID
df_sorted['next_year'] = df_sorted.groupby('ID')['period'].shift(-1)
df_sorted['next_premium'] = df_sorted.groupby('ID')['premium'].shift(-1)

# Keep only consecutive-year observations
renewal_df = df_sorted[
    df_sorted['next_year'] == df_sorted['period'] + 1
].copy()

# Rename current premium as previous premium
renewal_df = renewal_df.rename(
    columns={'premium': 'previous_premium'}
)

# Rename the next-year premium as the target
renewal_df = renewal_df.rename(
    columns={'next_premium': 'target_premium'}
)

print("Renewal pairs:", len(renewal_df))

Renewal pairs: 128257


In [11]:
print(
    renewal_df.groupby(['period', 'next_year'])
    .size()
)

print(
    "Missing target premiums:",
    renewal_df['target_premium'].isna().sum()
)

period  next_year
2017    2018.0       65818
2018    2019.0       62439
dtype: int64
Missing target premiums: 0


In [12]:
renewal_year_patterns = (
    renewal_df.groupby('ID')['period']
    .agg(lambda x: tuple(sorted(x.unique())))
)

renewal_year_patterns.value_counts()


period
(2017, 2018)    55635
(2017,)         10183
(2018,)          6804
Name: count, dtype: int64

In [13]:
renewal_df.columns

Index(['ID', 'ID_policy', 'ID_insured', 'period', 'date_effect_insured',
       'date_lapse_insured', 'date_effect_policy', 'date_lapse_policy',
       'year_effect_insured', 'year_lapse_insured', 'year_effect_policy',
       'year_lapse_policy', 'exposure_time', 'lapse', 'seniority_insured',
       'seniority_policy', 'type_policy', 'type_policy_dg', 'type_product',
       'reimbursement', 'new_business', 'distribution_channel', 'gender',
       'age', 'previous_premium', 'cost_claims_year', 'n_medical_services',
       'n_insured_pc', 'n_insured_mun', 'n_insured_prov', 'IICIMUN',
       'IICIPROV', 'C_H', 'C_GI', 'C_II', 'C_IE_P', 'C_IE_S', 'C_IE_T',
       'C_GE_P', 'C_GE_S', 'C_GE_T', 'C_C', 'next_year', 'target_premium'],
      dtype='object')

To verify whether these consecutive records represent genuine insurance renewal by anlysing various features and also analysing whether the customer have continued or discontinued their policy during the observation period

In [14]:
# Actual columns available in the main dataset
compare_cols = [
    'ID_policy', 'ID_insured',
    'type_policy', 'type_policy_dg',
    'type_product', 'reimbursement',
    'distribution_channel',
    'gender', 'age',
    'seniority_insured', 'seniority_policy',
    'exposure_time', 'lapse', 'new_business',
    'date_effect_insured', 'date_effect_policy',
    'date_lapse_insured', 'date_lapse_policy',
    'premium'
]

# Keep the original renewal_df unchanged
renewal_audit_df = renewal_df.copy()

# Retrieve characteristics from the target year
next_records = df[['ID', 'period'] + compare_cols].copy()

next_records['period'] = next_records['period'] - 1

next_records = next_records.rename(
    columns={col: f'next_{col}' for col in compare_cols}
)

renewal_audit_df = renewal_audit_df.merge(
    next_records,
    on=['ID', 'period'],
    how='left',
    validate='one_to_one'
)

print("Total renewal pairs:", len(renewal_audit_df))

Total renewal pairs: 128257


In [15]:
policy_features = [
    'ID_policy', 'ID_insured',
    'type_policy', 'type_policy_dg',
    'type_product', 'reimbursement',
    'distribution_channel'
]

policy_change_summary = []

for col in policy_features:
    old = renewal_audit_df[col]
    new = renewal_audit_df[f'next_{col}']

    valid = old.notna() & new.notna()
    changed = valid & old.ne(new)

    policy_change_summary.append({
        'Feature': col,
        'Comparable_Pairs': valid.sum(),
        'Changed_Count': changed.sum(),
        'Changed_Percentage': round(
            changed.sum() / valid.sum() * 100, 2
        ) if valid.sum() else np.nan
    })

policy_change_summary = pd.DataFrame(policy_change_summary)

display(policy_change_summary)

,Feature,Comparable_Pairs,Changed_Count,Changed_Percentage
0,ID_policy,128257,0,0.00
1,ID_insured,128257,0,0.00
2,type_policy,128257,221,0.17
3,type_policy_dg,128257,278,0.22
4,type_product,128257,0,0.00
5,reimbursement,128257,0,0.00
6,distribution_channel,128257,0,0.00


In [16]:
consistency_checks = pd.DataFrame({
    'Check': [
        'Gender changed',
        'Age did not increase by one',
        'Insured seniority decreased',
        'Policy seniority decreased'
    ],
    'Affected_Pairs': [
        (
            renewal_audit_df['gender'].notna()
            & renewal_audit_df['next_gender'].notna()
            & renewal_audit_df['gender'].ne(
                renewal_audit_df['next_gender']
            )
        ).sum(),

        (
            renewal_audit_df['age'].notna()
            & renewal_audit_df['next_age'].notna()
            & (
                renewal_audit_df['next_age']
                - renewal_audit_df['age'] != 1
            )
        ).sum(),

        (
            renewal_audit_df['next_seniority_insured']
            < renewal_audit_df['seniority_insured']
        ).sum(),

        (
            renewal_audit_df['next_seniority_policy']
            < renewal_audit_df['seniority_policy']
        ).sum()
    ]
})

display(consistency_checks)

,Check,Affected_Pairs
0,Gender changed,0
1,Age did not increase by one,4588
2,Insured seniority decreased,0
3,Policy seniority decreased,0


In [17]:
for col in ['lapse', 'new_business']:
    print(f"\n{col.upper()} — Previous vs Next Year")

    display(
        pd.crosstab(
            renewal_audit_df[col],
            renewal_audit_df[f'next_{col}'],
            dropna=False
        )
    )

exposure_comparison = renewal_audit_df[
    ['period', 'next_year', 'exposure_time', 'next_exposure_time']
].describe(include='all')

display(exposure_comparison)


LAPSE — Previous vs Next Year


next_lapse,1,2,3
lapse,,,
2,4042,106647,12628
3,4940,0,0



NEW_BUSINESS — Previous vs Next Year


next_new_business,No
new_business,
No,113556
Yes,14701


,period,next_year,exposure_time,next_exposure_time
count,128257.000000,128257.000000,128257.000000,128257.000000
mean,2017.486827,2018.486827,0.986067,0.943712
std,0.499828,0.499828,0.117213,0.218539
min,2017.000000,2018.000000,0.000000,0.000000
25%,2017.000000,2018.000000,1.000000,1.000000
50%,2017.000000,2018.000000,1.000000,1.000000
75%,2018.000000,2019.000000,1.000000,1.000000
max,2018.000000,2019.000000,1.000000,1.000000


In [18]:
date_cols = [
    'date_effect_insured',
    'date_effect_policy',
    'date_lapse_insured',
    'date_lapse_policy'
]

for col in date_cols:
    renewal_audit_df[col] = pd.to_datetime(
        renewal_audit_df[col], errors='coerce'
    )

    renewal_audit_df[f'next_{col}'] = pd.to_datetime(
        renewal_audit_df[f'next_{col}'], errors='coerce'
    )

date_summary = pd.DataFrame({
    'Feature': date_cols,

    'Previous_Year_Missing': [
        renewal_audit_df[col].isna().sum()
        for col in date_cols
    ],

    'Next_Year_Missing': [
        renewal_audit_df[f'next_{col}'].isna().sum()
        for col in date_cols
    ],

    'Changed_Count': [
        (
            renewal_audit_df[col].notna()
            & renewal_audit_df[f'next_{col}'].notna()
            & renewal_audit_df[col].ne(
                renewal_audit_df[f'next_{col}']
            )
        ).sum()
        for col in date_cols
    ]
})

display(date_summary)

,Feature,Previous_Year_Missing,Next_Year_Missing,Changed_Count
0,date_effect_insured,0,0,0
1,date_effect_policy,0,0,0
2,date_lapse_insured,103227,102163,7015
3,date_lapse_policy,106608,105587,5998


In [19]:
audit_flags = pd.DataFrame(index=renewal_audit_df.index)

audit_flags['policy_id_changed'] = (
    renewal_audit_df['ID_policy']
    != renewal_audit_df['next_ID_policy']
)

audit_flags['insured_id_changed'] = (
    renewal_audit_df['ID_insured']
    != renewal_audit_df['next_ID_insured']
)

audit_flags['product_changed'] = (
    renewal_audit_df['type_product'].notna()
    & renewal_audit_df['next_type_product'].notna()
    & renewal_audit_df['type_product'].ne(
        renewal_audit_df['next_type_product']
    )
)

audit_flags['reimbursement_changed'] = (
    renewal_audit_df['reimbursement'].notna()
    & renewal_audit_df['next_reimbursement'].notna()
    & renewal_audit_df['reimbursement'].ne(
        renewal_audit_df['next_reimbursement']
    )
)

audit_flags['partial_exposure_previous'] = (
    renewal_audit_df['exposure_time'] < 1
)

audit_flags['partial_exposure_next'] = (
    renewal_audit_df['next_exposure_time'] < 1
)

audit_flags['nonpositive_target_premium'] = (
    renewal_audit_df['target_premium'] <= 0
)

audit_flags['missing_target_premium'] = (
    renewal_audit_df['target_premium'].isna()
)

display(
    audit_flags.sum()
    .sort_values(ascending=False)
    .to_frame('Affected_Pairs')
)

,Affected_Pairs
partial_exposure_next,8982
partial_exposure_previous,1787
policy_id_changed,0
insured_id_changed,0
reimbursement_changed,0
product_changed,0
nonpositive_target_premium,0
missing_target_premium,0


nvestigate the 221 and 278 policy-type changes

In [20]:
policy_changed = renewal_audit_df[
    renewal_audit_df['type_policy'].ne(
        renewal_audit_df['next_type_policy']
    )
    |
    renewal_audit_df['type_policy_dg'].ne(
        renewal_audit_df['next_type_policy_dg']
    )
].copy()

policy_changed['premium_change_pct'] = (
    (policy_changed['target_premium'] -
     policy_changed['previous_premium'])
    / policy_changed['previous_premium'] * 100
)

display(
    policy_changed[
        [
            'ID', 'period', 'next_year',
            'type_policy', 'next_type_policy',
            'type_policy_dg', 'next_type_policy_dg',
            'previous_premium', 'target_premium',
            'premium_change_pct',
            'lapse', 'next_lapse',
            'exposure_time', 'next_exposure_time'
        ]
    ].head(20)
)

,ID,period,next_year,type_policy,next_type_policy,type_policy_dg,next_type_policy_dg,previous_premium,target_premium,premium_change_pct,lapse,next_lapse,exposure_time,next_exposure_time
354,10081_41,2017,2018.0,C,I,C1,I,540.60,714.00,32.075472,2,3,1.0,1.000000
355,10081_49,2017,2018.0,C,I,C1,I,591.86,744.37,25.767918,2,2,1.0,1.000000
357,10081_57,2017,2018.0,C,I,C1,I,384.54,473.28,23.076923,2,3,1.0,1.000000
358,10081_58,2017,2018.0,C,I,C1,I,380.77,468.64,23.076923,2,3,1.0,1.000000
359,10081_59,2017,2018.0,C,I,C1,I,540.60,744.37,37.693304,2,1,1.0,0.087557
2787,10749_1,2017,2018.0,C,I,C1,I,591.86,609.03,2.901024,2,2,1.0,1.000000
3087,10824_1,2017,2018.0,C,I,C1,I,2505.12,2555.30,2.003098,2,2,1.0,1.000000
3089,10824_2,2017,2018.0,C,I,C1,I,1587.72,1754.40,10.498073,2,2,1.0,1.000000
3091,10824_3,2017,2018.0,C,I,C1,I,696.90,718.11,3.043478,2,2,1.0,1.000000
3093,10825_1,2017,2018.0,C,I,C1,I,878.22,904.74,3.019744,2,2,1.0,1.000000


# Investigate partial exposure

In [21]:
exposure_audit = renewal_audit_df.copy()

exposure_audit['exposure_group'] = np.select(
    [
        (exposure_audit['exposure_time'] == 1)
        & (exposure_audit['next_exposure_time'] == 1),

        (exposure_audit['exposure_time'] < 1)
        & (exposure_audit['next_exposure_time'] == 1),

        (exposure_audit['exposure_time'] == 1)
        & (exposure_audit['next_exposure_time'] < 1),

        (exposure_audit['exposure_time'] < 1)
        & (exposure_audit['next_exposure_time'] < 1)
    ],
    [
        'Full → Full',
        'Partial → Full',
        'Full → Partial',
        'Partial → Partial'
    ],
    default='Other'
)

display(
    pd.crosstab(
        exposure_audit['exposure_group'],
        exposure_audit['next_lapse'],
        margins=True
    )
)

next_lapse,1,2,3,All
exposure_group,,,,
Full → Full,0,106647,10860,117507
Full → Partial,8963,0,0,8963
Partial → Full,0,0,1768,1768
Partial → Partial,19,0,0,19
All,8982,106647,12628,128257


4. Investigate the 4,588 age inconsistencies

In [22]:
renewal_audit_df['age_difference'] = (
    renewal_audit_df['next_age']
    - renewal_audit_df['age']
)

display(
    renewal_audit_df['age_difference']
    .value_counts(dropna=False)
    .sort_index()
)

age_difference
-30         1
-8          1
-3          3
-2          1
-1          1
 0       4557
 1     123669
 2          9
 3          2
 5          3
 9          2
 11         1
 13         1
 14         1
 18         1
 20         1
 21         2
 30         1
Name: count, dtype: int64

In [23]:
partial_lapse = renewal_audit_df[
    (renewal_audit_df['next_lapse'] == 1)
    & (renewal_audit_df['next_exposure_time'] > 0)
    & (renewal_audit_df['next_exposure_time'] < 1)
    & (renewal_audit_df['previous_premium'] > 0)
].copy()

partial_lapse['premium_ratio'] = (
    partial_lapse['target_premium']
    / partial_lapse['previous_premium']
)

partial_lapse['exposure_ratio'] = (
    partial_lapse['premium_ratio']
    / partial_lapse['next_exposure_time']
)

display(
    partial_lapse[
        [
            'previous_premium',
            'target_premium',
            'next_exposure_time',
            'premium_ratio',
            'exposure_ratio'
        ]
    ].describe()
)

,previous_premium,target_premium,next_exposure_time,premium_ratio,exposure_ratio
count,4741.000000,4741.000000,4741.000000,4741.000000,4741.000000
mean,844.690453,898.068994,0.371793,1.062705,65.065597
std,752.432658,797.168429,0.297599,0.121208,142.271302
min,34.680000,34.986972,0.002626,0.530743,0.909126
25%,522.170000,546.093533,0.087557,1.020452,1.747589
50%,692.580000,742.560000,0.331279,1.040874,3.236990
75%,962.880000,1021.110000,0.583333,1.080169,11.821481
max,19482.000000,19859.630000,0.997146,4.661460,719.003927


In [24]:
zero_exposure = renewal_audit_df[
    (renewal_audit_df['next_lapse'] == 1)
    & (renewal_audit_df['next_exposure_time'] == 0)
].copy()

print("Zero-exposure records:", len(zero_exposure))

display(
    zero_exposure[
        [
            'ID', 'period', 'next_year',
            'previous_premium', 'target_premium',
            'lapse', 'next_lapse',
            'exposure_time', 'next_exposure_time',
            'next_date_lapse_insured',
            'next_date_lapse_policy'
        ]
    ].head(20)
)

Zero-exposure records: 4241


,ID,period,next_year,previous_premium,target_premium,lapse,next_lapse,exposure_time,next_exposure_time,next_date_lapse_insured,next_date_lapse_policy
32,10005_1,2018,2019.0,978.18,956.8110,3,1,1.0,0.0,2019-01-01,2019-01-01
251,10058_1,2018,2019.0,832.24,836.8355,3,1,1.0,0.0,2019-01-01,2019-01-01
253,10058_2,2018,2019.0,697.68,693.0620,3,1,1.0,0.0,2019-01-01,2019-01-01
255,10058_3,2018,2019.0,552.84,545.6020,3,1,1.0,0.0,2019-01-01,2019-01-01
257,10058_4,2018,2019.0,466.62,469.0980,3,1,1.0,0.0,2019-01-01,2019-01-01
313,10075_1,2018,2019.0,1095.85,726.2405,3,1,1.0,0.0,2019-01-01,2019-01-01
315,10075_2,2018,2019.0,1095.85,726.2405,3,1,1.0,0.0,2019-01-01,2019-01-01
317,10075_3,2018,2019.0,766.59,527.1695,3,1,1.0,0.0,2019-01-01,2019-01-01
319,10075_4,2018,2019.0,795.88,527.1695,3,1,1.0,0.0,2019-01-01,2019-01-01
375,10086_1,2018,2019.0,1525.10,1518.9840,3,1,1.0,0.0,2019-01-01,2019-01-01


In [25]:
display(
    zero_exposure[
        [
            'period',
            'next_year',
            'next_date_lapse_insured',
            'next_date_lapse_policy'
        ]
    ].head(20)
)

print("Missing insured lapse dates:",
      zero_exposure['next_date_lapse_insured'].isna().sum())

print("Missing policy lapse dates:",
      zero_exposure['next_date_lapse_policy'].isna().sum())

print("\nTarget-year distribution:")
display(zero_exposure['next_year'].value_counts())

,period,next_year,next_date_lapse_insured,next_date_lapse_policy
32,2018,2019.0,2019-01-01,2019-01-01
251,2018,2019.0,2019-01-01,2019-01-01
253,2018,2019.0,2019-01-01,2019-01-01
255,2018,2019.0,2019-01-01,2019-01-01
257,2018,2019.0,2019-01-01,2019-01-01
313,2018,2019.0,2019-01-01,2019-01-01
315,2018,2019.0,2019-01-01,2019-01-01
317,2018,2019.0,2019-01-01,2019-01-01
319,2018,2019.0,2019-01-01,2019-01-01
375,2018,2019.0,2019-01-01,2019-01-01


Missing insured lapse dates: 0
Missing policy lapse dates: 0

Target-year distribution:


next_year
2019.0    2256
2018.0    1985
Name: count, dtype: int64

In [26]:
target_start = pd.to_datetime(
    zero_exposure['next_year'].astype(int).astype(str) + '-01-01'
)

insured_jan1 = (
    zero_exposure['next_date_lapse_insured'] == target_start
)

policy_jan1 = (
    zero_exposure['next_date_lapse_policy'] == target_start
)

print("Total zero-exposure records:", len(zero_exposure))
print("Insured lapsed on January 1:", insured_jan1.sum())
print("Policy lapsed on January 1:", policy_jan1.sum())
print("Both dates January 1:", (insured_jan1 & policy_jan1).sum())
print("Other date combinations:", (~(insured_jan1 & policy_jan1)).sum())

Total zero-exposure records: 4241
Insured lapsed on January 1: 4222
Policy lapsed on January 1: 4212
Both dates January 1: 4212
Other date combinations: 29


In [27]:
    unusual_zero_exposure = zero_exposure.loc[
        ~(insured_jan1 & policy_jan1)
    ].copy()

    display(
        unusual_zero_exposure[
            [
                'ID', 'period', 'next_year',
                'previous_premium', 'target_premium',
                'exposure_time', 'next_exposure_time',
                'next_lapse',
                'next_date_lapse_insured',
                'next_date_lapse_policy'
            ]
        ]
    )

,ID,period,next_year,previous_premium,target_premium,exposure_time,next_exposure_time,next_lapse,next_date_lapse_insured,next_date_lapse_policy
2484,10671_2,2017,2018.0,625.26,637.31,0.0,0.0,1,2018-05-01,2019-01-01
8217,12079_1,2017,2018.0,1138.27,1175.64,0.0,0.0,1,2018-11-14,2019-01-01
11178,12747_10,2017,2018.0,927.18,1053.66,0.0,0.0,1,2018-06-01,2019-01-01
11183,12747_5,2017,2018.0,1013.03,1043.33,0.0,0.0,1,2018-06-01,2019-01-01
11184,12747_7,2017,2018.0,1110.78,1143.42,0.0,0.0,1,2018-06-01,2019-01-01
11185,12747_8,2017,2018.0,1329.06,1355.42,0.0,0.0,1,2018-06-01,2019-01-01
11186,12747_9,2017,2018.0,1329.06,1368.84,0.0,0.0,1,2018-06-01,2019-01-01
31267,18181_2,2017,2018.0,1038.28,1132.20,0.0,0.0,1,2018-03-18,2019-01-01
31386,18211_6,2017,2018.0,902.94,939.30,0.0,0.0,1,2018-07-01,2019-01-01
44580,22210_1,2017,2018.0,675.24,729.30,0.0,0.0,1,2018-04-09,2019-01-01


In [28]:
lapse_3_to_1 = renewal_audit_df[
    (renewal_audit_df['lapse'] == 3) &
    (renewal_audit_df['next_lapse'] == 1)
].copy()

print("Total 3 → 1 transitions:", len(lapse_3_to_1))

Total 3 → 1 transitions: 4940


In [29]:
import pandas as pd

lapse_3_to_1['previous_exposure_group'] = (
    lapse_3_to_1['exposure_time'].eq(0).map({
        True: 'Zero',
        False: 'Positive'
    })
)

lapse_3_to_1['next_exposure_group'] = (
    lapse_3_to_1['next_exposure_time'].eq(0).map({
        True: 'Zero',
        False: 'Positive'
    })
)

display(
    pd.crosstab(
        lapse_3_to_1['previous_exposure_group'],
        lapse_3_to_1['next_exposure_group'],
        margins=True
    )
)

next_exposure_group,Positive,Zero,All
previous_exposure_group,,,
Positive,718,4222,4940
All,718,4222,4940


In [30]:
display(
    lapse_3_to_1[
        [
            'ID',
            'period',
            'next_year',
            'previous_premium',
            'target_premium',
            'lapse',
            'next_lapse',
            'exposure_time',
            'next_exposure_time',
            'date_lapse_insured',
            'next_date_lapse_insured',
            'date_lapse_policy',
            'next_date_lapse_policy'
        ]
    ].head(20)
)

,ID,period,next_year,previous_premium,target_premium,lapse,next_lapse,exposure_time,next_exposure_time,date_lapse_insured,next_date_lapse_insured,date_lapse_policy,next_date_lapse_policy
32,10005_1,2018,2019.0,978.18,956.8110,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
165,10044_2,2017,2018.0,858.50,931.2600,3,1,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
251,10058_1,2018,2019.0,832.24,836.8355,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
253,10058_2,2018,2019.0,697.68,693.0620,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
255,10058_3,2018,2019.0,552.84,545.6020,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
257,10058_4,2018,2019.0,466.62,469.0980,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
313,10075_1,2018,2019.0,1095.85,726.2405,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
315,10075_2,2018,2019.0,1095.85,726.2405,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
317,10075_3,2018,2019.0,766.59,527.1695,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01
319,10075_4,2018,2019.0,795.88,527.1695,3,1,1.0,0.000000,2018-12-31,2019-01-01,2018-12-31,2019-01-01


In [31]:
print(
    "Positive target-year exposure:",
    (lapse_3_to_1['next_exposure_time'] > 0).sum()
)

print(
    "Zero target-year exposure:",
    (lapse_3_to_1['next_exposure_time'] == 0).sum()
)

Positive target-year exposure: 718
Zero target-year exposure: 4222


In [32]:
previous_year_end = pd.to_datetime(
    lapse_3_to_1['period'].astype(int).astype(str) + '-12-31'
)

previous_insured_lapse = pd.to_datetime(
    lapse_3_to_1['date_lapse_insured'],
    errors='coerce'
)

previous_policy_lapse = pd.to_datetime(
    lapse_3_to_1['date_lapse_policy'],
    errors='coerce'
)

print(
    "Insured lapse on previous year-end:",
    (previous_insured_lapse == previous_year_end).sum()
)

print(
    "Policy lapse on previous year-end:",
    (previous_policy_lapse == previous_year_end).sum()
)

print(
    "Missing previous insured lapse dates:",
    previous_insured_lapse.isna().sum()
)

print(
    "Missing previous policy lapse dates:",
    previous_policy_lapse.isna().sum()
)

Insured lapse on previous year-end: 4940
Policy lapse on previous year-end: 4212
Missing previous insured lapse dates: 0
Missing previous policy lapse dates: 604


In [33]:
positive_3_to_1 = lapse_3_to_1[
    lapse_3_to_1['next_exposure_time'] > 0
].copy()

print("Records:", len(positive_3_to_1))

display(
    positive_3_to_1[
        [
            'ID',
            'period',
            'next_year',
            'previous_premium',
            'target_premium',
            'exposure_time',
            'next_exposure_time',
            'date_lapse_insured',
            'next_date_lapse_insured',
            'date_lapse_policy',
            'next_date_lapse_policy'
        ]
    ].head(20)
)

print("\nTarget-year exposure summary:")
display(
    positive_3_to_1['next_exposure_time'].describe()
)

Records: 718


,ID,period,next_year,previous_premium,target_premium,exposure_time,next_exposure_time,date_lapse_insured,next_date_lapse_insured,date_lapse_policy,next_date_lapse_policy
165,10044_2,2017,2018.0,858.50,931.2600,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
687,10180_2,2017,2018.0,792.54,777.7000,1.0,0.002626,2017-12-31,2018-01-01,2019-12-31,2019-12-31
688,10180_3,2017,2018.0,591.60,580.3800,1.0,0.002626,2017-12-31,2018-01-01,2019-12-31,2019-12-31
1305,1033_2,2017,2018.0,1189.78,1229.1000,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
1909,10506_4,2017,2018.0,538.56,553.4800,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
3334,10887_1,2017,2018.0,769.08,897.6000,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
3335,10887_12,2017,2018.0,534.48,612.0600,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
3336,10887_13,2017,2018.0,447.78,517.1400,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
3337,10887_14,2017,2018.0,443.39,517.1400,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT
3556,10945_1,2017,2018.0,885.36,1168.9200,1.0,0.002626,2017-12-31,2018-01-01,NaT,NaT



Target-year exposure summary:


count    7.180000e+02
mean     2.625571e-03
std      8.679664e-19
min      2.625571e-03
25%      2.625571e-03
50%      2.625571e-03
75%      2.625571e-03
max      2.625571e-03
Name: next_exposure_time, dtype: float64

In [34]:
next_lapse_date = pd.to_datetime(
    positive_3_to_1['next_date_lapse_insured'],
    errors='coerce'
)

target_year = positive_3_to_1['next_year'].astype(int)

within_target_year = (
    next_lapse_date.dt.year == target_year
)

print("Lapse within target year:", within_target_year.sum())
print("Lapse outside target year:", (~within_target_year & next_lapse_date.notna()).sum())
print("Missing lapse dates:", next_lapse_date.isna().sum())

Lapse within target year: 718
Lapse outside target year: 0
Missing lapse dates: 0


In [35]:
target_start = pd.to_datetime(
    positive_3_to_1['next_year'].astype(int).astype(str) + '-01-01'
)

actual_lapse = pd.to_datetime(
    positive_3_to_1['next_date_lapse_insured'],
    errors='coerce'
)

print(
    "Lapsed exactly January 1:",
    (actual_lapse == target_start).sum()
)

print(
    "Lapsed after January 1:",
    (actual_lapse > target_start).sum()
)

print(
    "Other dates:",
    (actual_lapse < target_start).sum()
)

Lapsed exactly January 1: 718
Lapsed after January 1: 0
Other dates: 0


In [36]:
import pandas as pd
import numpy as np

# Identify zero-exposure observations
zero_exposure_mask = (
    renewal_audit_df['next_exposure_time'] == 0
)

# Identify the 718 start-of-year lapse observations
target_start = pd.to_datetime(
    renewal_audit_df['next_year'].astype(int).astype(str) + '-01-01'
)

next_lapse_date = pd.to_datetime(
    renewal_audit_df['next_date_lapse_insured'],
    errors='coerce'
)

jan1_lapse_mask = (
    (renewal_audit_df['lapse'] == 3) &
    (renewal_audit_df['next_lapse'] == 1) &
    (renewal_audit_df['next_exposure_time'] > 0) &
    (next_lapse_date == target_start)
)

# Create the provisional eligible renewal dataset
eligible_renewals_df = renewal_audit_df.loc[
    ~(zero_exposure_mask | jan1_lapse_mask)
].copy()

print("Original pairs:", len(renewal_audit_df))
print("Zero-exposure excluded:", zero_exposure_mask.sum())
print("January 1 minimal-exposure excluded:", jan1_lapse_mask.sum())
print("Remaining pairs:", len(eligible_renewals_df))

assert len(eligible_renewals_df) == 123298

Original pairs: 128257
Zero-exposure excluded: 4241
January 1 minimal-exposure excluded: 718
Remaining pairs: 123298


In [37]:
policy_changes = eligible_renewals_df[
    (eligible_renewals_df['type_policy'] != eligible_renewals_df['next_type_policy']) |
    (eligible_renewals_df['type_policy_dg'] != eligible_renewals_df['next_type_policy_dg'])
].copy()

print("Total policy-change records:", len(policy_changes))

print("\nPolicy type transitions:")
display(
    policy_changes.groupby(
        ['type_policy', 'next_type_policy']
    ).size().sort_values(ascending=False)
)

print("\nDetailed policy type transitions:")
display(
    policy_changes.groupby(
        ['type_policy_dg', 'next_type_policy_dg']
    ).size().sort_values(ascending=False)
)

Total policy-change records: 268

Policy type transitions:


type_policy  next_type_policy
C            I                   182
I            I                    55
             C                    31
dtype: int64


Detailed policy type transitions:


type_policy_dg  next_type_policy_dg
C1              I                      182
I               SE                      42
                C1                      31
SE              I                       13
dtype: int64

In [38]:
# Percentage change in premium
eligible_renewals_df['premium_change_pct'] = (
    (
        eligible_renewals_df['target_premium']
        - eligible_renewals_df['previous_premium']
    )
    / eligible_renewals_df['previous_premium']
) * 100

# Flag policy-type changes
eligible_renewals_df['policy_type_changed'] = (
    (eligible_renewals_df['type_policy'] !=
     eligible_renewals_df['next_type_policy'])
    |
    (eligible_renewals_df['type_policy_dg'] !=
     eligible_renewals_df['next_type_policy_dg'])
)

# Compare premium changes
display(
    eligible_renewals_df.groupby('policy_type_changed')[
        'premium_change_pct'
    ].describe()
)

,count,mean,std,min,25%,50%,75%,max
policy_type_changed,,,,,,,,
False,123030.0,5.601308,8.612262,-73.053529,2.163087,4.488872,7.951070,366.145978
True,268.0,9.544894,14.219967,-34.027845,2.157006,6.320715,16.021267,85.740125


In [39]:
import pandas as pd
import numpy as np

audit = renewal_audit_df.copy()

# Convert dates safely
next_lapse_date = pd.to_datetime(
    audit['next_date_lapse_insured'],
    errors='coerce'
)

target_year_start = pd.to_datetime(
    audit['next_year'].astype(int).astype(str) + '-01-01'
)

# Rule 1: Exclude zero target-year exposure
zero_exposure = audit['next_exposure_time'].eq(0)

# Rule 2: Exclude January 1 lapses with minimal exposure
jan1_minimal_exposure = (
    audit['lapse'].eq(3)
    & audit['next_lapse'].eq(1)
    & audit['next_exposure_time'].gt(0)
    & next_lapse_date.eq(target_year_start)
)

# Combine exclusions
exclude = zero_exposure | jan1_minimal_exposure

final_audit_df = audit.loc[~exclude].copy()

print("Original renewal pairs:", len(audit))
print("Zero-exposure exclusions:", zero_exposure.sum())
print("January 1 minimal-exposure exclusions:",
      jan1_minimal_exposure.sum())
print("Final renewal pairs:", len(final_audit_df))

assert len(final_audit_df) == 123298

Original renewal pairs: 128257
Zero-exposure exclusions: 4241
January 1 minimal-exposure exclusions: 718
Final renewal pairs: 123298


In [40]:
# Remove future-year audit variables
future_columns = [
    col for col in final_audit_df.columns
    if col.startswith('next_')
]

# Keep the prediction target separately
final_renewal_df = final_audit_df.drop(
    columns=future_columns + [
        'premium_change_pct',
        'policy_type_changed'
    ],
    errors='ignore'
).copy()

# Keep target_premium as the supervised learning target
print("Final shape:", final_renewal_df.shape)
print("Target column:", 'target_premium')

display(final_renewal_df.head())

Final shape: (123298, 44)
Target column: target_premium


,ID,ID_policy,ID_insured,period,date_effect_insured,date_lapse_insured,date_effect_policy,date_lapse_policy,year_effect_insured,year_lapse_insured,...,C_II,C_IE_P,C_IE_S,C_IE_T,C_GE_P,C_GE_S,C_GE_T,C_C,target_premium,age_difference
0,10000_1,10000,1,2017,2004-07-01,NaT,2004-07-01,NaT,2004,NaN,...,18.0,67.0,91.0,17.0,23.0,72.0,70.0,C3,630.3600,1
1,10000_1,10000,1,2018,2004-07-01,NaT,2004-07-01,NaT,2004,NaN,...,13.0,66.0,90.0,18.0,23.0,72.0,70.0,C3,652.5858,1
2,10001_1,10001,1,2017,2004-07-09,NaT,2004-07-09,NaT,2004,NaN,...,98.0,7.0,4.0,97.0,1.0,1.0,100.0,C1,2882.5400,1
3,10001_1,10001,1,2018,2004-07-09,NaT,2004-07-09,NaT,2004,NaN,...,98.0,7.0,4.0,97.0,1.0,1.0,100.0,C1,3038.3931,1
4,10002_1,10002,1,2017,2004-08-01,NaT,2004-08-01,NaT,2004,NaN,...,90.0,24.0,20.0,80.0,6.0,2.0,99.0,C1,2010.9100,1


In [41]:
print("Total observations:", len(final_renewal_df))

print("\nYear-wise distribution:")
display(
    final_renewal_df.groupby(
        ['period']
    ).size()
)

print("\nDuplicate ID-year pairs:")
print(
    final_renewal_df.duplicated(
        subset=['ID', 'period']
    ).sum()
)

print("\nMissing target premiums:")
print(
    final_renewal_df['target_premium'].isna().sum()
)

print("\nNon-positive target premiums:")
print(
    (final_renewal_df['target_premium'] <= 0).sum()
)

print("\nRemaining future-year audit columns:")
print([
    col for col in final_renewal_df.columns
    if col.startswith('next_')
])

Total observations: 123298

Year-wise distribution:


period
2017    63467
2018    59831
dtype: int64


Duplicate ID-year pairs:
0

Missing target premiums:
0

Non-positive target premiums:
0

Remaining future-year audit columns:
[]


In [42]:
final_renewal_df.to_csv(
    "final_renewal_dataset.csv",
    index=False
)

print("Saved: final_renewal_dataset.csv")

Saved: final_renewal_dataset.csv


In [43]:
final_renewal_df = pd.read_csv("final_renewal_dataset.csv")

print("Dataset shape:", final_renewal_df.shape)
print("Total columns:", len(final_renewal_df.columns))

print("\nAge difference distribution:")
print(final_renewal_df["age_difference"].value_counts(dropna=False).sort_index())

Dataset shape: (123298, 44)
Total columns: 44

Age difference distribution:
age_difference
-30         1
-8          1
-3          3
-2          1
-1          1
 0       4168
 1     119101
 2          8
 3          2
 5          3
 9          2
 11         1
 13         1
 14         1
 18         1
 21         2
 30         1
Name: count, dtype: int64


In [44]:
final_renewal_dataset = final_renewal_df[
    final_renewal_df["age_difference"].isin([0, 1])
].copy()

final_renewal_dataset.reset_index(drop=True, inplace=True)

print("Before cleaning:", len(final_renewal_df))
print("After cleaning:", len(final_renewal_dataset))
print("Rows removed:", len(final_renewal_df) - len(final_renewal_dataset))

print(
    "Remaining unusual age differences:",
    (~final_renewal_dataset["age_difference"].isin([0, 1])).sum()
)

Before cleaning: 123298
After cleaning: 123269
Rows removed: 29
Remaining unusual age differences: 0


In [45]:
print("Final observations:", len(final_renewal_dataset))

print(
    "Unusual age differences:",
    (~final_renewal_dataset["age_difference"].isin([0, 1])).sum()
)

Final observations: 123269
Unusual age differences: 0


final_renewal_dataset.to_csv(
    "cleaned.csv",
    index=False
)

print("Saved: cleaned.csv")
print("Dataset shape:", final_renewal_dataset.shape)

Final Feature Integrity Check

In [46]:
candidate_features = [
    "age",
    "gender",
    "seniority_insured",
    "seniority_policy",
    "type_policy_dg",
    "type_product",
    "reimbursement",
    "distribution_channel",
    "previous_premium",
    "cost_claims_year",
    "n_medical_services"
]

columns = candidate_features + ["target_premium"]

results = []

for col in columns:
    s = final_renewal_dataset[col]

    results.append({
        "Feature": col,
        "Data Type": str(s.dtype),
        "Missing Values": s.isna().sum(),
        "Infinite Values": (
            np.isinf(s).sum()
            if pd.api.types.is_numeric_dtype(s)
            else 0
        ),
        "Unique Values": s.nunique(dropna=True)
    })

integrity_report = pd.DataFrame(results)

display(integrity_report)

,Feature,Data Type,Missing Values,Infinite Values,Unique Values
0,age,int64,0,0,100
1,gender,object,0,0,2
2,seniority_insured,int64,0,0,86
3,seniority_policy,int64,0,0,87
4,type_policy_dg,object,0,0,6
5,type_product,object,0,0,4
6,reimbursement,object,0,0,2
7,distribution_channel,object,0,0,3
8,previous_premium,float64,0,0,5833
9,cost_claims_year,float64,0,0,9272


In [47]:
categorical_features = [
    "gender",
    "type_policy_dg",
    "type_product",
    "reimbursement",
    "distribution_channel"
]

comparison = final_renewal_dataset[
    ["ID", "period"] + categorical_features
].merge(
    df[["ID", "period"] + categorical_features],
    on=["ID", "period"],
    how="left",
    suffixes=("_renewal", "_original"),
    validate="one_to_one",
    indicator=True
)

print("Unmatched records:",
      (comparison["_merge"] != "both").sum())

for col in categorical_features:
    mismatches = (
        comparison[f"{col}_renewal"].fillna("__MISSING__")
        != comparison[f"{col}_original"].fillna("__MISSING__")
    ).sum()

    print(f"{col}: {mismatches} mismatches")

Unmatched records: 0
gender: 0 mismatches
type_policy_dg: 0 mismatches
type_product: 0 mismatches
reimbursement: 0 mismatches
distribution_channel: 0 mismatches


In [48]:
excluded_pairs = final_renewal_df.iloc[0:0].copy()

print("Available checkpoint rows:", len(final_renewal_df))
print("Current cleaned rows:", len(final_renewal_dataset))

print("\nExposure distribution in cleaned dataset:")
print(
    final_renewal_dataset["exposure_time"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nRenewal pairs by source year:")
print(
    final_renewal_dataset["period"]
    .value_counts()
    .sort_index()
)

Available checkpoint rows: 123298
Current cleaned rows: 123269

Exposure distribution in cleaned dataset:
exposure_time
1.0    121501
0.0      1768
Name: count, dtype: int64

Renewal pairs by source year:
period
2017    63455
2018    59814
Name: count, dtype: int64


In [49]:
checks = {
    "Total observations": len(final_renewal_dataset),

    "Duplicate ID-period pairs":
        final_renewal_dataset.duplicated(
            subset=["ID", "period"]
        ).sum(),

    "Missing target premiums":
        final_renewal_dataset["target_premium"].isna().sum(),

    "Non-positive target premiums":
        (final_renewal_dataset["target_premium"] <= 0).sum(),

    "Non-positive previous premiums":
        (final_renewal_dataset["previous_premium"] <= 0).sum(),

    "Negative ages":
        (final_renewal_dataset["age"] < 0).sum(),

    "Unusual age differences":
        (~final_renewal_dataset["age_difference"].isin([0, 1])).sum(),

    "Negative claim costs":
        (final_renewal_dataset["cost_claims_year"] < 0).sum(),

    "Negative medical services":
        (final_renewal_dataset["n_medical_services"] < 0).sum(),

    "Invalid reporting years":
        (~final_renewal_dataset["period"].isin([2017, 2018])).sum()
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["Validation Check", "Result"]
    )
)

,Validation Check,Result
0,Total observations,123269
1,Duplicate ID-period pairs,0
2,Missing target premiums,0
3,Non-positive target premiums,0
4,Non-positive previous premiums,0
5,Negative ages,0
6,Unusual age differences,0
7,Negative claim costs,0
8,Negative medical services,0
9,Invalid reporting years,0


In [50]:
final_renewal_dataset.to_csv(
    "cleaned.csv",
    index=False
)

print("Saved: cleaned.csv")
print("Final shape:", final_renewal_dataset.shape)

Saved: cleaned.csv
Final shape: (123269, 44)


In [51]:
# Research Question:
# Why are lapse = 3 observations absent from the cleaned dataset?
#
# Purpose:
# Track lapse statuses through renewal-pair construction and cleaning.
# Check whether lapse = 3 records have a consecutive-year target.

# 1. Lapse distribution across cleaning stages
stages = {
    "Original Dataset": df,
    "Renewal Pairs": renewal_df,
    "After Exposure/Lapse Cleaning": final_audit_df,
    "Final Cleaned Dataset": final_renewal_dataset
}

lapse_summary = pd.DataFrame({
    stage: data["lapse"].value_counts()
    for stage, data in stages.items()
}).fillna(0).astype(int)

print("1. Lapse status across cleaning stages:")
display(lapse_summary)


# 2. Check whether original lapse = 3 records have a next-year record
original_sorted = df.sort_values(
    ["ID", "period"]
).copy()

original_sorted["next_period"] = (
    original_sorted.groupby("ID")["period"].shift(-1)
)

original_sorted["next_premium"] = (
    original_sorted.groupby("ID")["premium"].shift(-1)
)

original_sorted["has_next_year"] = (
    original_sorted["next_period"]
    == original_sorted["period"] + 1
)

print("\n2. Consecutive-year availability by lapse status:")
display(
    pd.crosstab(
        original_sorted["lapse"],
        original_sorted["has_next_year"],
        margins=True
    )
)


# 3. Specifically investigate lapse = 3
lapse3 = original_sorted.loc[
    original_sorted["lapse"].eq(3)
].copy()

print("\n3. Lapse = 3 records by year:")
display(
    pd.crosstab(
        lapse3["period"],
        lapse3["has_next_year"],
        margins=True
    )
)


# 4. Verify whether lapse = 3 records have valid next-year premiums
print("\n4. Lapse = 3 target availability:")

display(
    pd.Series({
        "Total lapse = 3 records": len(lapse3),
        "Have consecutive-year record":
            lapse3["has_next_year"].sum(),
        "Have consecutive-year record and premium":
            (
                lapse3["has_next_year"]
                & lapse3["next_premium"].notna()
            ).sum(),
        "No consecutive-year record":
            (~lapse3["has_next_year"]).sum()
    }).to_frame("Records")
)


# 5. Check exposure distribution for lapse = 3
print("\n5. Exposure distribution for lapse = 3:")
display(
    lapse3["exposure_time"].value_counts(dropna=False)
)


# 6. Check lapse transitions among all original renewal pairs
next_lapse = original_sorted.groupby("ID")["lapse"].shift(-1)

transition_df = original_sorted.loc[
    original_sorted["has_next_year"],
    ["lapse"]
].copy()

transition_df["next_lapse"] = next_lapse.loc[
    original_sorted["has_next_year"]
].values

print("\n6. Current-year to next-year lapse transitions:")
display(
    pd.crosstab(
        transition_df["lapse"],
        transition_df["next_lapse"],
        margins=True
    )
)

1. Lapse status across cleaning stages:


,Original Dataset,Renewal Pairs,After Exposure/Lapse Cleaning,Final Cleaned Dataset
lapse,,,,
1,16348,0,0,0
2,187271,123317,123298,123269
3,25092,4940,0,0



2. Consecutive-year availability by lapse status:


has_next_year,False,True,All
lapse,,,
1,16348,0,16348
2,63954,123317,187271
3,20152,4940,25092
All,100454,128257,228711



3. Lapse = 3 records by year:


has_next_year,False,True,All
period,,,
2017,7035,2332,9367
2018,6536,2608,9144
2019,6581,0,6581
All,20152,4940,25092


... (2 outputs left out)


5. Exposure distribution for lapse = 3:


exposure_time
1.0    25092
Name: count, dtype: int64


6. Current-year to next-year lapse transitions:


next_lapse,1.0,2.0,3.0,All
lapse,,,,
2,4042,106647,12628,123317
3,4940,0,0,4940
All,8982,106647,12628,128257


In [52]:
# Research Question:
# Why were 4,940 lapse = 3 renewal pairs excluded?
#
# Purpose:
# Examine next-year exposure, premium validity,
# and the exact cleaning exclusion conditions.
# No plots or modifications to existing dataframes.

audit_lapse3 = renewal_audit_df.loc[
    renewal_audit_df["lapse"].eq(3)
].copy()

print("1. Total lapse = 3 renewal pairs:")
print(len(audit_lapse3))

print("\n2. Next-year exposure distribution:")
display(
    audit_lapse3["next_exposure_time"]
    .value_counts(dropna=False)
)

print("\n3. Next-year lapse distribution:")
display(
    audit_lapse3["next_lapse"]
    .value_counts(dropna=False)
)

print("\n4. Premium validity:")
display(pd.Series({
    "Positive previous premium":
        audit_lapse3["previous_premium"].gt(0).sum(),
    "Positive target premium":
        audit_lapse3["target_premium"].gt(0).sum(),
    "Missing target premium":
        audit_lapse3["target_premium"].isna().sum()
}).to_frame("Records"))

print("\n5. Next-year exposure vs lapse:")
display(
    pd.crosstab(
        audit_lapse3["next_exposure_time"],
        audit_lapse3["next_lapse"],
        margins=True
    )
)

1. Total lapse = 3 renewal pairs:
4940

2. Next-year exposure distribution:


next_exposure_time
0.000000    4222
0.002626     718
Name: count, dtype: int64


3. Next-year lapse distribution:


next_lapse
1    4940
Name: count, dtype: int64


4. Premium validity:


,Records
Positive previous premium,4940
Positive target premium,4940
Missing target premium,0



5. Next-year exposure vs lapse:


next_lapse,1,All
next_exposure_time,,
0.0,4222,4222
0.002626,718,718
All,4940,4940


In [53]:
# Research Question:
# Why did most lapse 2 → 1 renewal pairs survive cleaning?
#
# Purpose:
# Audit next-year exposure, exclusion conditions, and
# premium validity without modifying existing dataframes.

# Step 1: Select the 4,042 transitions
audit_2_to_1 = renewal_audit_df.loc[
    renewal_audit_df["lapse"].eq(2)
    & renewal_audit_df["next_lapse"].eq(1)
].copy()

print("1. Total lapse 2 → 1 transitions:")
print(len(audit_2_to_1))


# Step 2: Examine next-year exposure
print("\n2. Next-year exposure distribution:")
display(
    audit_2_to_1["next_exposure_time"]
    .value_counts(dropna=False)
    .sort_index()
)


# Step 3: Reconstruct the exact exclusion conditions
next_lapse_date = pd.to_datetime(
    audit_2_to_1["next_date_lapse_insured"],
    errors="coerce"
)

target_year_start = pd.to_datetime(
    audit_2_to_1["next_year"].astype(int).astype(str)
    + "-01-01"
)

zero_exposure = (
    audit_2_to_1["next_exposure_time"].eq(0)
)

jan1_minimal_exposure = (
    audit_2_to_1["lapse"].eq(3)
    & audit_2_to_1["next_lapse"].eq(1)
    & audit_2_to_1["next_exposure_time"].gt(0)
    & next_lapse_date.eq(target_year_start)
)

audit_2_to_1["excluded"] = (
    zero_exposure | jan1_minimal_exposure
)

print("\n3. Exclusion rule results:")
display(
    audit_2_to_1["excluded"]
    .value_counts(dropna=False)
)

print("\n4. Next-year exposure vs exclusion:")
display(
    pd.crosstab(
        audit_2_to_1["next_exposure_time"],
        audit_2_to_1["excluded"],
        margins=True
    )
)


# Step 4: Verify premium validity
print("\n5. Premium validity:")
display(
    pd.Series({
        "Positive previous premium":
            audit_2_to_1["previous_premium"].gt(0).sum(),
        "Positive target premium":
            audit_2_to_1["target_premium"].gt(0).sum(),
        "Missing target premium":
            audit_2_to_1["target_premium"].isna().sum()
    }).to_frame("Records")
)


# Step 5: Compare retained and excluded observations
print("\n6. Premium statistics by exclusion status:")
display(
    audit_2_to_1.groupby("excluded")[
        ["previous_premium", "target_premium", "next_exposure_time"]
    ].agg(["count", "mean", "median", "min", "max"])
    .round(3)
)


# Step 6: Inspect representative observations
print("\n7. Sample retained observations:")
display(
    audit_2_to_1.loc[
        ~audit_2_to_1["excluded"],
        [
            "ID", "period", "lapse", "next_lapse",
            "next_exposure_time",
            "previous_premium", "target_premium"
        ]
    ].head(10)
)

1. Total lapse 2 → 1 transitions:
4042

2. Next-year exposure distribution:


next_exposure_time
0.000000    19
0.005365     6
0.008105    11
0.010845     4
0.013584     5
            ..
0.977968     1
0.980708     3
0.986187     1
0.988927     1
0.997146    19
Name: count, Length: 302, dtype: int64


3. Exclusion rule results:


excluded
False    4023
True       19
Name: count, dtype: int64


4. Next-year exposure vs exclusion:


excluded,False,True,All
next_exposure_time,,,
0.0,0,19,19
0.005365,6,0,6
0.008105,11,0,11
0.010845,4,0,4
0.013584,5,0,5
...,...,...,...
0.980708,3,0,3
0.986187,1,0,1
0.988927,1,0,1


... (2 outputs left out)


6. Premium statistics by exclusion status:


previous_premium                                    target_premium  \
                    count     mean  median     min       max          count   
excluded                                                                      
False                4023  845.182  691.56   34.68  13303.72           4023   
True                   19  823.283  723.18  427.38   1329.06             19   

                                              next_exposure_time         \
             mean  median      min        max              count   mean   
excluded                                                                  
False     900.709  742.56   34.987  12816.014               4023  0.438   
True      880.981  744.60  459.550   1368.840                 19  0.000   

                               
         median    min    max  
excluded                       
False     0.416  0.005  0.997  
True      0.000  0.000  0.000


7. Sample retained observations:


,ID,period,lapse,next_lapse,next_exposure_time,previous_premium,target_premium
7,10002_3,2018,2,1,0.087557,1230.18,849.968656
106,10029_2,2017,2,1,0.903995,4056.54,4137.970000
281,10062_1,2018,2,1,0.750457,602.82,617.313843
350,10080_1,2017,2,1,0.750457,1152.60,1187.280000
351,10080_2,2017,2,1,0.750457,1141.30,1175.640000
352,10080_3,2017,2,1,0.750457,791.84,816.080000
353,10080_4,2017,2,1,0.750457,799.68,816.080000
359,10081_59,2017,2,1,0.087557,540.60,744.370000
361,10082_1,2018,2,1,0.750457,602.82,617.313843
363,10082_2,2018,2,1,0.750457,381.78,398.700690


In [55]:
# Research Question:
# Why does the cleaned dataset contain only lapse = 2?
#
# Purpose:
# Audit lapse-status exclusions across cleaning stages.
# Identify whether records were excluded during renewal matching
# or subsequent cleaning.
#
# No plots. No modification to existing DataFrames.

# --------------------------------------------------
# 1. Lapse distribution across cleaning stages
# --------------------------------------------------

stages = {
    "Original": df,
    "Renewal Pairs": renewal_df,
    "After Exposure/Lapse Cleaning": final_audit_df,
    "Final Cleaned": final_renewal_dataset
}

lapse_summary = pd.DataFrame({
    name: data["lapse"].value_counts()
    for name, data in stages.items()
}).fillna(0).astype(int)

print("1. Lapse distribution across cleaning stages:")
display(lapse_summary)


# --------------------------------------------------
# 2. Check consecutive-year availability
# --------------------------------------------------

audit = df.sort_values(["ID", "period"]).copy()

audit["next_period"] = (
    audit.groupby("ID")["period"].shift(-1)
)

audit["next_premium"] = (
    audit.groupby("ID")["premium"].shift(-1)
)

audit["next_lapse"] = (
    audit.groupby("ID")["lapse"].shift(-1)
)

audit["has_next_year"] = (
    audit["next_period"].eq(audit["period"] + 1)
)

print("\n2. Consecutive-year availability by lapse:")
display(
    pd.crosstab(
        audit["lapse"],
        audit["has_next_year"],
        margins=True
    )
)


# --------------------------------------------------
# 3. Investigate lapse = 3
# --------------------------------------------------

lapse3 = audit[audit["lapse"].eq(3)].copy()

print("\n3. Lapse = 3 records by year:")
display(
    pd.crosstab(
        lapse3["period"],
        lapse3["has_next_year"],
        margins=True
    )
)

print("\n4. Lapse = 3 target availability:")
display(
    pd.Series({
        "Total lapse = 3 records": len(lapse3),
        "Consecutive-year records":
            int(lapse3["has_next_year"].sum()),
        "Consecutive-year records with valid premium":
            int((
                lapse3["has_next_year"]
                & lapse3["next_premium"].notna()
            ).sum()),
        "Without consecutive-year record":
            int((~lapse3["has_next_year"]).sum())
    }).to_frame("Count")
)


# --------------------------------------------------
# 4. Exposure and lapse relationships
# --------------------------------------------------

print("\n5. Exposure distribution for lapse = 3:")
display(
    lapse3["exposure_time"].value_counts(dropna=False)
)

print("\n6. Current-year to next-year lapse transitions:")

valid_pairs = audit.loc[
    audit["has_next_year"]
].copy()

display(
    pd.crosstab(
        valid_pairs["lapse"],
        valid_pairs["next_lapse"],
        margins=True
    )
)


# --------------------------------------------------
# 5. Verify exclusion during later cleaning
# --------------------------------------------------

print("\n7. Records removed at each stage:")

stage_counts = pd.DataFrame({
    "Stage": [
        "Original Dataset",
        "Renewal Pairs",
        "After Exposure/Lapse Cleaning",
        "Final Cleaned Dataset"
    ],
    "Records": [
        len(df),
        len(renewal_df),
        len(final_audit_df),
        len(final_renewal_dataset)
    ]
})

stage_counts["Removed From Previous Stage"] = (
    stage_counts["Records"].shift(1)
    - stage_counts["Records"]
)

display(stage_counts)

1. Lapse distribution across cleaning stages:


,Original,Renewal Pairs,After Exposure/Lapse Cleaning,Final Cleaned
lapse,,,,
1,16348,0,0,0
2,187271,123317,123298,123269
3,25092,4940,0,0



2. Consecutive-year availability by lapse:


has_next_year,False,True,All
lapse,,,
1,16348,0,16348
2,63954,123317,187271
3,20152,4940,25092
All,100454,128257,228711



3. Lapse = 3 records by year:


has_next_year,False,True,All
period,,,
2017,7035,2332,9367
2018,6536,2608,9144
2019,6581,0,6581
All,20152,4940,25092


... (4 outputs left out)


6. Current-year to next-year lapse transitions:


next_lapse,1.0,2.0,3.0,All
lapse,,,,
2,4042,106647,12628,123317
3,4940,0,0,4940
All,8982,106647,12628,128257



7. Records removed at each stage:


,Stage,Records,Removed From Previous Stage
0,Original Dataset,228711,NaN
1,Renewal Pairs,128257,100454.0
2,After Exposure/Lapse Cleaning,123298,4959.0
3,Final Cleaned Dataset,123269,29.0


In [56]:
# Research Question:
# Why were 4,940 lapse = 3 renewal pairs excluded?
#
# Purpose:
# Verify their next-year exposure, lapse status, and premiums.
# No changes to existing DataFrames.

lapse3_pairs = renewal_audit_df.loc[
    renewal_audit_df["lapse"].eq(3)
].copy()

print("1. Total lapse = 3 renewal pairs:")
print(len(lapse3_pairs))

print("\n2. Next-year exposure distribution:")
display(
    lapse3_pairs["next_exposure_time"]
    .value_counts(dropna=False)
)

print("\n3. Next-year lapse distribution:")
display(
    lapse3_pairs["next_lapse"]
    .value_counts(dropna=False)
)

print("\n4. Next-year premium validity:")
display(pd.Series({
    "Positive target premium":
        lapse3_pairs["target_premium"].gt(0).sum(),
    "Zero target premium":
        lapse3_pairs["target_premium"].eq(0).sum(),
    "Missing target premium":
        lapse3_pairs["target_premium"].isna().sum()
}).to_frame("Records"))

print("\n5. Sample excluded observations:")
display(
    lapse3_pairs[
        [
            "ID",
            "period",
            "lapse",
            "next_lapse",
            "next_exposure_time",
            "previous_premium",
            "target_premium"
        ]
    ].head(10)
)

1. Total lapse = 3 renewal pairs:
4940

2. Next-year exposure distribution:


next_exposure_time
0.000000    4222
0.002626     718
Name: count, dtype: int64


3. Next-year lapse distribution:


next_lapse
1    4940
Name: count, dtype: int64


4. Next-year premium validity:


,Records
Positive target premium,4940
Zero target premium,0
Missing target premium,0



5. Sample excluded observations:


,ID,period,lapse,next_lapse,next_exposure_time,previous_premium,target_premium
32,10005_1,2018,3,1,0.000000,978.18,956.8110
165,10044_2,2017,3,1,0.002626,858.50,931.2600
251,10058_1,2018,3,1,0.000000,832.24,836.8355
253,10058_2,2018,3,1,0.000000,697.68,693.0620
255,10058_3,2018,3,1,0.000000,552.84,545.6020
257,10058_4,2018,3,1,0.000000,466.62,469.0980
313,10075_1,2018,3,1,0.000000,1095.85,726.2405
315,10075_2,2018,3,1,0.000000,1095.85,726.2405
317,10075_3,2018,3,1,0.000000,766.59,527.1695
319,10075_4,2018,3,1,0.000000,795.88,527.1695


In [58]:
# Research Question:
# Is the cleaned dataset consistent and suitable for ML preprocessing?
#
# Purpose:
# Check missing values, duplicates, premium validity,
# exposure values, and year-wise consistency.
# Do not modify the original dataset.

audit = final_renewal_dataset.copy()

# 1. Dataset dimensions
print("1. Dataset dimensions:")
print("Rows:", audit.shape[0])
print("Columns:", audit.shape[1])

# 2. Missing values
print("\n2. Missing values:")

missing = pd.DataFrame({
    "Missing Count": audit.isna().sum(),
    "Missing %": audit.isna().mean() * 100
})

display(
    missing[missing["Missing Count"] > 0]
    .sort_values("Missing Count", ascending=False)
)

# 3. Duplicate records
print("\n3. Duplicate audit:")

display(pd.Series({
    "Exact duplicate rows": audit.duplicated().sum(),
    "Duplicate ID-period pairs":
        audit.duplicated(["ID", "period"]).sum(),
    "Unique IDs": audit["ID"].nunique()
}).to_frame("Count"))

# 4. Premium validity
print("\n4. Premium validity:")

display(pd.Series({
    "Missing previous premium":
        audit["previous_premium"].isna().sum(),
    "Missing target premium":
        audit["target_premium"].isna().sum(),
    "Previous premium <= 0":
        audit["previous_premium"].le(0).sum(),
    "Target premium <= 0":
        audit["target_premium"].le(0).sum()
}).to_frame("Count"))

# 5. Exposure validity
print("\n5. Exposure validity:")

display(pd.Series({
    "Exposure < 0": audit["exposure_time"].lt(0).sum(),
    "Exposure > 1": audit["exposure_time"].gt(1).sum(),
    "Exposure = 0": audit["exposure_time"].eq(0).sum(),
    "Exposure = 1": audit["exposure_time"].eq(1).sum(),
    "Partial exposure":
        audit["exposure_time"]
        .between(0, 1, inclusive="neither").sum()
}).to_frame("Count"))

# 6. Year-wise consistency
print("\n6. Year-wise dataset summary:")

display(
    audit.groupby("period")
    .agg(
        Records=("ID", "size"),
        Unique_IDs=("ID", "nunique"),
        Median_Previous_Premium=("previous_premium", "median"),
        Median_Target_Premium=("target_premium", "median")
    ).round(2)
)

# 7. Constant columns
print("\n7. Constant columns:")

constant_cols = audit.columns[
    audit.nunique(dropna=False) <= 1
].tolist()

print(constant_cols)

# 8. Final dataset completeness
print("\n8. Complete observations:")

display(pd.Series({
    "Complete rows": audit.notna().all(axis=1).sum(),
    "Rows with missing values": audit.isna().any(axis=1).sum(),
    "Total rows": len(audit)
}).to_frame("Count"))

1. Dataset dimensions:
Rows: 123269
Columns: 44

2. Missing values:


,Missing Count,Missing %
date_lapse_policy,105976,85.971331
year_lapse_policy,105976,85.971331
date_lapse_insured,103199,83.718534
year_lapse_insured,103199,83.718534
C_GI,7415,6.015300
C_IE_S,7415,6.015300
C_IE_P,7415,6.015300
C_II,7415,6.015300
C_GE_P,7415,6.015300
C_GE_S,7415,6.015300



3. Duplicate audit:


,Count
Exact duplicate rows,0
Duplicate ID-period pairs,0
Unique IDs,69737



4. Premium validity:


,Count
Missing previous premium,0
Missing target premium,0
Previous premium <= 0,0
Target premium <= 0,0


... (2 outputs left out)


6. Year-wise dataset summary:


,Records,Unique_IDs,Median_Previous_Premium,Median_Target_Premium
period,,,,
2017,63455,63455,776.69,813.05
2018,59814,59814,799.92,853.74



7. Constant columns:
['lapse']

8. Complete observations:


,Count
Complete rows,15186
Rows with missing values,108083
Total rows,123269


In [59]:
# Research Question:
# Which features are eligible for next-year premium prediction?
#
# Purpose:
# Audit all columns for uniqueness, missingness, data types,
# constants, and potential leakage.
# Do not modify the original dataframe.

audit = final_renewal_dataset.copy()

# --------------------------------------------------
# 1. Complete feature inventory
# --------------------------------------------------

feature_audit = pd.DataFrame({
    "Feature": audit.columns,
    "Data Type": audit.dtypes.astype(str).values,
    "Missing Count": audit.isna().sum().values,
    "Missing %": (
        audit.isna().mean() * 100
    ).round(2).values,
    "Unique Values": audit.nunique(dropna=True).values
})

feature_audit["Unique %"] = (
    feature_audit["Unique Values"]
    / len(audit) * 100
).round(2)

display(feature_audit)


# --------------------------------------------------
# 2. Identify known leakage and identifier candidates
# --------------------------------------------------

known_groups = {
    "Target": ["target_premium"],
    "Future-derived": ["age_difference"],
    "Identifiers": ["ID", "ID_policy", "ID_insured"],
    "Constant": ["lapse"],
    "Historical premium": ["previous_premium"]
}

print("\n2. Known feature groups:")

for group, cols in known_groups.items():
    present = [col for col in cols if col in audit.columns]
    print(f"{group}: {present}")


# --------------------------------------------------
# 3. Identify date-related features
# --------------------------------------------------

date_features = [
    col for col in audit.columns
    if "date" in col.lower()
    or "year" in col.lower()
]

print("\n3. Date-related features:")
display(
    feature_audit[
        feature_audit["Feature"].isin(date_features)
    ]
)


# --------------------------------------------------
# 4. High-cardinality features
# --------------------------------------------------

print("\n4. High-cardinality features (>90% unique):")

display(
    feature_audit[
        feature_audit["Unique %"] > 90
    ]
)


# --------------------------------------------------
# 5. Constant and near-constant features
# --------------------------------------------------

print("\n5. Constant features:")

display(
    feature_audit[
        feature_audit["Unique Values"] <= 1
    ]
)


# --------------------------------------------------
# 6. Missingness among candidate predictors
# --------------------------------------------------

print("\n6. Features with missing values:")

display(
    feature_audit[
        feature_audit["Missing Count"] > 0
    ].sort_values(
        "Missing %",
        ascending=False
    )
)


# --------------------------------------------------
# 7. Final feature count
# --------------------------------------------------

print("\n7. Total columns:")
print(len(audit.columns))

print("\n8. Column names:")
print(audit.columns.tolist())

,Feature,Data Type,Missing Count,Missing %,Unique Values,Unique %
0,ID,object,0,0.00,69737,56.57
1,ID_policy,int64,0,0.00,32543,26.40
2,ID_insured,int64,0,0.00,1205,0.98
3,period,int64,0,0.00,2,0.00
4,date_effect_insured,object,0,0.00,4822,3.91
5,date_lapse_insured,object,103199,83.72,437,0.35
6,date_effect_policy,object,0,0.00,3034,2.46
7,date_lapse_policy,object,105976,85.97,257,0.21
8,year_effect_insured,int64,0,0.00,86,0.07
9,year_lapse_insured,float64,103199,83.72,2,0.00



2. Known feature groups:
Target: ['target_premium']
Future-derived: ['age_difference']
Identifiers: ['ID', 'ID_policy', 'ID_insured']
Constant: ['lapse']
Historical premium: ['previous_premium']

3. Date-related features:


,Feature,Data Type,Missing Count,Missing %,Unique Values,Unique %
4,date_effect_insured,object,0,0.00,4822,3.91
5,date_lapse_insured,object,103199,83.72,437,0.35
6,date_effect_policy,object,0,0.00,3034,2.46
7,date_lapse_policy,object,105976,85.97,257,0.21
8,year_effect_insured,int64,0,0.00,86,0.07
9,year_lapse_insured,float64,103199,83.72,2,0.00
10,year_effect_policy,int64,0,0.00,87,0.07
11,year_lapse_policy,float64,105976,85.97,2,0.00
25,cost_claims_year,float64,0,0.00,9272,7.52



4. High-cardinality features (>90% unique):


,Feature,Data Type,Missing Count,Missing %,Unique Values,Unique %



5. Constant features:


,Feature,Data Type,Missing Count,Missing %,Unique Values,Unique %
13,lapse,int64,0,0.0,1,0.0



6. Features with missing values:


,Feature,Data Type,Missing Count,Missing %,Unique Values,Unique %
7,date_lapse_policy,object,105976,85.97,257,0.21
11,year_lapse_policy,float64,105976,85.97,2,0.00
5,date_lapse_insured,object,103199,83.72,437,0.35
9,year_lapse_insured,float64,103199,83.72,2,0.00
33,C_GI,float64,7415,6.02,100,0.08
36,C_IE_S,float64,7415,6.02,100,0.08
35,C_IE_P,float64,7415,6.02,100,0.08
34,C_II,float64,7415,6.02,100,0.08
38,C_GE_P,float64,7415,6.02,100,0.08
39,C_GE_S,float64,7415,6.02,100,0.08



7. Total columns:
44

8. Column names:
['ID', 'ID_policy', 'ID_insured', 'period', 'date_effect_insured', 'date_lapse_insured', 'date_effect_policy', 'date_lapse_policy', 'year_effect_insured', 'year_lapse_insured', 'year_effect_policy', 'year_lapse_policy', 'exposure_time', 'lapse', 'seniority_insured', 'seniority_policy', 'type_policy', 'type_policy_dg', 'type_product', 'reimbursement', 'new_business', 'distribution_channel', 'gender', 'age', 'previous_premium', 'cost_claims_year', 'n_medical_services', 'n_insured_pc', 'n_insured_mun', 'n_insured_prov', 'IICIMUN', 'IICIPROV', 'C_H', 'C_GI', 'C_II', 'C_IE_P', 'C_IE_S', 'C_IE_T', 'C_GE_P', 'C_GE_S', 'C_GE_T', 'C_C', 'target_premium', 'age_difference']


In [60]:
# Research Question:
# Are the effective dates, effective years, and seniority
# features consistent and potentially redundant?
#
# Purpose:
# Audit date consistency, seniority calculations,
# future effective dates, and feature redundancy.
# Do not modify the original dataframe.

import pandas as pd
import numpy as np

audit = final_renewal_dataset.copy()

# --------------------------------------------------
# 1. Convert effective dates for auditing
# --------------------------------------------------

date_columns = [
    "date_effect_insured",
    "date_effect_policy"
]

for col in date_columns:
    audit[col] = pd.to_datetime(
        audit[col],
        errors="coerce"
    )

print("1. Effective date parsing:")

display(pd.DataFrame({
    "Feature": date_columns,
    "Unparseable Dates": [
        audit[col].isna().sum()
        for col in date_columns
    ]
}))


# --------------------------------------------------
# 2. Effective date vs effective year
# --------------------------------------------------

print("\n2. Effective date-year consistency:")

date_year_checks = []

for entity in ["insured", "policy"]:

    date_col = f"date_effect_{entity}"
    year_col = f"year_effect_{entity}"

    valid = (
        audit[date_col].notna()
        & audit[year_col].notna()
    )

    mismatches = (
        audit.loc[valid, date_col].dt.year
        != audit.loc[valid, year_col]
    ).sum()

    date_year_checks.append({
        "Entity": entity,
        "Valid Comparisons": int(valid.sum()),
        "Year Mismatches": int(mismatches),
        "Mismatch %": round(
            mismatches / valid.sum() * 100, 4
        ) if valid.sum() else np.nan
    })

display(pd.DataFrame(date_year_checks))


# --------------------------------------------------
# 3. Seniority consistency
# --------------------------------------------------

print("\n3. Seniority consistency:")

seniority_checks = []

for entity in ["insured", "policy"]:

    year_col = f"year_effect_{entity}"
    seniority_col = f"seniority_{entity}"

    # Expected seniority using calendar-year difference
    expected = audit["period"] - audit[year_col]

    difference = audit[seniority_col] - expected

    seniority_checks.append({
        "Entity": entity,
        "Exact Matches": int((difference == 0).sum()),
        "Difference +1": int((difference == 1).sum()),
        "Difference -1": int((difference == -1).sum()),
        "Other Differences": int(
            (~difference.isin([0, 1, -1])).sum()
        ),
        "Minimum Difference": difference.min(),
        "Maximum Difference": difference.max()
    })

display(pd.DataFrame(seniority_checks))


# --------------------------------------------------
# 4. Future effective dates
# --------------------------------------------------

print("\n4. Future effective-date audit:")

future_checks = []

for entity in ["insured", "policy"]:

    date_col = f"date_effect_{entity}"
    year_col = f"year_effect_{entity}"

    # Compare against December 31 of observation year
    observation_year_end = pd.to_datetime(
        audit["period"].astype(str) + "-12-31"
    )

    future_date = (
        audit[date_col] > observation_year_end
    )

    future_year = (
        audit[year_col] > audit["period"]
    )

    future_checks.append({
        "Entity": entity,
        "Effective Date After Observation Year": int(
            future_date.sum()
        ),
        "Effective Year After Observation Year": int(
            future_year.sum()
        )
    })

display(pd.DataFrame(future_checks))


# --------------------------------------------------
# 5. Redundancy between effective year and seniority
# --------------------------------------------------

print("\n5. Effective year vs seniority correlation:")

redundancy_checks = []

for entity in ["insured", "policy"]:

    year_col = f"year_effect_{entity}"
    seniority_col = f"seniority_{entity}"

    redundancy_checks.append({
        "Entity": entity,
        "Pearson Correlation": audit[
            [year_col, seniority_col]
        ].corr().iloc[0, 1]
    })

display(pd.DataFrame(redundancy_checks).round(6))


# --------------------------------------------------
# 6. Sample inconsistent observations
# --------------------------------------------------

print("\n6. Seniority inconsistencies (sample):")

for entity in ["insured", "policy"]:

    year_col = f"year_effect_{entity}"
    seniority_col = f"seniority_{entity}"

    expected = audit["period"] - audit[year_col]

    inconsistent = audit.loc[
        audit[seniority_col] != expected,
        [
            "ID",
            "period",
            f"date_effect_{entity}",
            year_col,
            seniority_col
        ]
    ].copy()

    print(f"\n{entity.upper()}: {len(inconsistent)} inconsistencies")

    display(inconsistent.head(10))

1. Effective date parsing:


,Feature,Unparseable Dates
0,date_effect_insured,0
1,date_effect_policy,0



2. Effective date-year consistency:


,Entity,Valid Comparisons,Year Mismatches,Mismatch %
0,insured,123269,0,0.0
1,policy,123269,0,0.0



3. Seniority consistency:


,Entity,Exact Matches,Difference +1,Difference -1,Other Differences,Minimum Difference,Maximum Difference
0,insured,123269,0,0,0,0,0
1,policy,123269,0,0,0,0,0


... (4 outputs left out)


6. Seniority inconsistencies (sample):

INSURED: 0 inconsistencies


,ID,period,date_effect_insured,year_effect_insured,seniority_insured



POLICY: 0 inconsistencies


,ID,period,date_effect_policy,year_effect_policy,seniority_policy


In [61]:
# Research Question:
# Are contextual and portfolio-level features temporally
# consistent and potentially available at prediction time?
#
# Purpose:
# Audit missingness, year-wise availability, repeated-ID
# consistency, and relationships with portfolio counts.
# Do not modify the original dataset.

import pandas as pd
import numpy as np

audit = final_renewal_dataset.copy()

context_cols = [
    "IICIMUN", "IICIPROV",
    "C_H", "C_GI", "C_II",
    "C_IE_P", "C_IE_S", "C_IE_T",
    "C_GE_P", "C_GE_S", "C_GE_T",
    "C_C",
    "n_insured_pc", "n_insured_mun", "n_insured_prov"
]

# 1. Year-wise missingness
print("1. Year-wise missingness (%):")

display(
    audit.groupby("period")[context_cols]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
    .T
)

# 2. Year-wise unique values
print("\n2. Unique values by observation year:")

display(
    audit.groupby("period")[context_cols]
    .nunique()
    .T
)

# 3. Same-ID changes between 2017 and 2018
print("\n3. Feature changes for IDs present in both years:")

paired = audit[
    audit["ID"].duplicated(keep=False)
].copy()

change_results = []

for col in context_cols:
    wide = paired.pivot(
        index="ID",
        columns="period",
        values=col
    )

    if not {2017, 2018}.issubset(wide.columns):
        continue

    both_present = wide[2017].notna() & wide[2018].notna()
    changed = (
        wide.loc[both_present, 2017]
        != wide.loc[both_present, 2018]
    )

    change_results.append({
        "Feature": col,
        "IDs Compared": int(both_present.sum()),
        "Changed": int(changed.sum()),
        "Changed %": round(changed.mean() * 100, 2)
        if len(changed) else np.nan
    })

display(pd.DataFrame(change_results))

# 4. Shared missingness patterns
print("\n4. Missingness pattern groups:")

missing_groups = {}

for col in context_cols:
    pattern = tuple(audit.index[audit[col].isna()])
    missing_groups.setdefault(pattern, []).append(col)

for cols in missing_groups.values():
    print(cols)

# 5. Portfolio count consistency
print("\n5. Portfolio count summary by year:")

count_cols = [
    "n_insured_pc",
    "n_insured_mun",
    "n_insured_prov"
]

display(
    audit.groupby("period")[count_cols]
    .agg(["min", "median", "max"])
)

1. Year-wise missingness (%):


period,2017,2018
IICIMUN,6.06,3.62
IICIPROV,3.61,1.17
C_H,6.06,3.62
C_GI,7.22,4.74
C_II,7.22,4.74
C_IE_P,7.22,4.74
C_IE_S,7.22,4.74
C_IE_T,7.22,4.74
C_GE_P,7.22,4.74
C_GE_S,7.22,4.74



2. Unique values by observation year:


period,2017,2018
IICIMUN,640,600
IICIPROV,41,42
C_H,6,6
C_GI,100,100
C_II,100,100
C_IE_P,100,100
C_IE_S,100,100
C_IE_T,100,100
C_GE_P,100,100
C_GE_S,100,100



3. Feature changes for IDs present in both years:


,Feature,IDs Compared,Changed,Changed %
0,IICIMUN,51520,50449,97.92
1,IICIPROV,52884,52758,99.76
2,C_H,51520,200,0.39
3,C_GI,50865,38835,76.35
4,C_II,50865,43630,85.78
5,C_IE_P,50865,10038,19.73
6,C_IE_S,50865,26394,51.89
7,C_IE_T,50865,20073,39.46
8,C_GE_P,50865,1851,3.64
9,C_GE_S,50865,1446,2.84



4. Missingness pattern groups:
['IICIMUN', 'C_H']
['IICIPROV']
['C_GI', 'C_II', 'C_IE_P', 'C_IE_S', 'C_IE_T', 'C_GE_P', 'C_GE_S', 'C_GE_T']
['C_C']
['n_insured_pc', 'n_insured_mun', 'n_insured_prov']

5. Portfolio count summary by year:


n_insured_pc              n_insured_mun                n_insured_prov  \
                min median   max           min  median    max            min   
period                                                                         
2017              1  131.0  2518             1  1234.0  19315              6   
2018              1  111.0  2441             1  1043.0  17991              4   

                       
        median    max  
period                 
2017    7467.0  29465  
2018    7013.0  27261